# videospectra quickstart

This notebook runs the full **videospectra** pipeline end-to-end on synthetic frames — no GPU, no webcam, no torch.

We generate 50 RGB frames with a scripted scene change at frame 25 (blue → red), feed them through a `Session` configured with the built-in `ColorHistogramEmbedder`, and plot the spectral signals (entropy, motion, anomaly) with shot boundaries marked.

Expected runtime: ~10 s on the free Colab tier.

Repo: https://github.com/hdubey-debug/videospectra

In [ ]:
# Install videospectra from GitHub if it is not already importable.
# (Skipped automatically when running locally with a dev install.)
import importlib.util
if importlib.util.find_spec("videospectra") is None:
    %pip install -q git+https://github.com/hdubey-debug/videospectra.git

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from videospectra.analytics.spectral import SpectralConfig
from videospectra.embedders import ColorHistogramEmbedder
from videospectra.events import AnomalyAlert, FrameMetrics, ShotBoundary
from videospectra.session import Session
from videospectra.sinks import MemorySink
from videospectra.types import Frame

In [ ]:
def make_synthetic_frames(n=50, size=64, scene_change_at=25, seed=0):
    """Generate n RGB frames with a hard scene change at `scene_change_at`.

    Scene 0 (frames 0..scene_change_at-1): blue-dominant gradient + noise.
    Scene 1 (frames scene_change_at..n-1): red-dominant gradient + noise.
    The color flip produces a visible shot-boundary and anomaly spike;
    the small per-pixel noise keeps motion_score non-zero within a scene.
    """
    rng = np.random.default_rng(seed)
    frames = []
    for i in range(n):
        scene = 0 if i < scene_change_at else 1
        t = (i if scene == 0 else (i - scene_change_at)) / max(1, scene_change_at)
        if scene == 0:
            r, g, b = int(40 + 20 * t), int(60 + 20 * t), int(200 - 20 * t)
        else:
            r, g, b = int(220 - 20 * t), int(60 + 20 * t), int(40 + 20 * t)
        base = np.zeros((size, size, 3), dtype=np.uint8)
        base[..., 0], base[..., 1], base[..., 2] = r, g, b
        noise = rng.integers(-10, 11, size=(size, size, 3), dtype=np.int16)
        arr = np.clip(base.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        frames.append(Image.fromarray(arr))
    return frames

frames = make_synthetic_frames()
print(f"Generated {len(frames)} frames; scene change at frame 25")

In [ ]:
async def run_demo():
    sink = MemorySink()
    session = Session(
        frame_embedder=ColorHistogramEmbedder.make_image(),
        spectral_config=SpectralConfig(window_frames=10),
        sinks=[sink],
        source_fps=2.0,
    )
    await session.start()
    for i, img in enumerate(frames):
        await session.process_frame(Frame.from_pil(img, source_id="synth", frame_id=i))
    await session.aclose()

    events = []
    async for event in sink:
        events.append(event)
    return events

events = await run_demo()
print(f"Collected {len(events)} events")

In [ ]:
frame_id, entropy, motion, anomaly = [], [], [], []
shot_frames, alert_frames = [], []
for e in events:
    if isinstance(e, FrameMetrics):
        frame_id.append(e.frame_id)
        entropy.append(e.payload.entropy_norm)
        motion.append(e.payload.motion_score)
        anomaly.append(e.payload.anomaly_score)
    elif isinstance(e, ShotBoundary):
        shot_frames.append(e.frame_id)
    elif isinstance(e, AnomalyAlert):
        alert_frames.append(e.frame_id)

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, y, label in zip(
    axes,
    [entropy, motion, anomaly],
    ["Entropy (normalized)", "Motion score", "Anomaly score"],
):
    ax.plot(frame_id, y, linewidth=1.5)
    for j, sf in enumerate(shot_frames):
        ax.axvline(
            sf,
            color="red",
            linestyle="--",
            alpha=0.7,
            label="shot boundary" if j == 0 else None,
        )
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
axes[0].set_title("videospectra spectral signals — synthetic scene change at frame 25")
axes[-1].set_xlabel("frame_id")
if shot_frames:
    axes[0].legend(loc="upper right")
plt.tight_layout()
plt.show()

print(f"Shot boundaries detected at frames: {shot_frames}")
print(f"Anomaly alerts at frames:           {alert_frames}")